In [2]:
# !pip install wandb
!nvidia-smi

Wed Jul 23 13:50:56 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 566.03                 Driver Version: 566.03         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   44C    P8              3W /  130W |    1164MiB /   8188MiB |      3%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [20]:
import wandb

# export WANDB_API_KEY=7855589e123f4caee4595a43d22c86cce82fbda1
wandb.login(key="7855589e123f4caee4595a43d22c86cce82fbda1",relogin=True)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\mza0288\_netrc


True

In [25]:
from ultralytics import YOLO
import torch
import wandb

# ✅ Initialize wandb
wandb.init(
    project="yolo-training",     # change to your project name
    name="yolov11n-run_1",      # optional run name
    config={
        "model": "yolo11m-seg.pt",
        "epochs": 100,
        "optimizer": "SGD",
        "seed": 7,
        "image_size": 640,       # default in YOLOv8
        "device": "cuda" if torch.cuda.is_available() else "cpu"
    }
)

# ✅ Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

try:
    # Load model
    model = YOLO('yolo11n.pt').to(device)

    # Train with W&B logging enabled
    results = model.train(
        data="C:/data/data_rice/humay-ai-rice-diseases/data.yaml",
        epochs=100,
        device=device,
        seed=7,
        optimizer="SGD",
        show_boxes=False,
        show_labels=False,
        imgsz=640,                 # set image size explicitly if needed
        project="runs/train",      # local logging dir
        name="yolo11m-sgd-run_2",    # experiment name
        save=True,
        verbose=True
        # wandb is auto-enabled if wandb is installed
    )

    # ✅ Optional: Log final model as W&B artifact
    artifact = wandb.Artifact('yolo11m-final', type='model')
    artifact.add_file(f"runs/train/yolo11m-sgd-run/weights/best.pt")
    wandb.log_artifact(artifact)

except Exception as e:
    print(f"An error occurred: {e}")

# ✅ End wandb run
wandb.finish()


Using device: cuda
New https://pypi.org/project/ultralytics/8.3.170 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.94  Python-3.9.20 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: task=detect, mode=train, model=yolo11n.pt, data=C:/data/data_rice/humay-ai-rice-diseases/data.yaml, epochs=100, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=cuda, workers=8, project=runs/train, name=yolo11m-sgd-run_2, exist_ok=False, pretrained=True, optimizer=SGD, verbose=True, seed=7, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms

train: Scanning C:\data\data_rice\humay-ai-rice-diseases\train\labels.cache... 34445 images, 0 backgrounds, 0 corrupt: Scanning C:\data\data_rice\humay-ai-rice-diseases\train\labels.cache... 34445 images, 0 backgrounds, 0 corrupt: 
val: Scanning C:\data\data_rice\humay-ai-rice-diseases\valid\labels.cache... 4284 images, 0 backgrounds, 0 corrupt: 100Scanning C:\data\data_rice\humay-ai-rice-diseases\valid\labels.cache... 4284 images, 0 backgrounds, 0 corrupt: 100


Plotting labels to runs\train\yolo11m-sgd-run_2\labels.jpg... 
optimizer: SGD(lr=0.01, momentum=0.937) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
TensorBoard: model graph visualization added 
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to runs\train\yolo11m-sgd-run_2
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      1/100      2.57G      1.883      2.838      1.837         54        640: 100%|██████████| 2153/2153 [05:43<00:00,
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95):   0%|          | 0/134 [00:05


An error occurred: DataLoader worker (pid(s) 47136, 47904, 45012, 47152, 45656, 49044, 26408) exited unexpectedly


In [19]:
wandb.finish()